# Adversarial BERT — Full Dataset Training with Cross-Validation

This notebook trains the proposed AdversarialBERT model on each of the five
fake-news datasets using 5-fold cross-validation on the full dataset (no subsets).

## Workflow
1. Imports & environment setup.
2. Define dataset paths (original + pre-computed perturbations).
3. Define Dataset classes, model architecture, custom trainer.
4. 85/15 stratified train/test split → 5-fold CV on the 85% training pool.
5. Save the best model (by validation F1) per dataset.
6. Cross-dataset generalisation evaluation (Accuracy, F1, Precision, Recall).

# Section 1: Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import torch
import os
import random
from torch import nn
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    BertTokenizer, BertModel, BertForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, TrainerCallback
)
from transformers.modeling_outputs import SequenceClassifierOutput

from google.colab import drive
drive.mount('/content/drive')

try:
    import torch_xla as torch_xla_pkg
    import torch_xla.core.xla_model as xm
    if not hasattr(torch, "xla"):
        torch.xla = torch_xla_pkg
    _TORCH_XLA_AVAILABLE = True
except Exception:
    xm = None
    _TORCH_XLA_AVAILABLE = False

# Reproducibility
SEED = random.randint(0, 4294967295)
print(f"Random seed: {SEED}")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Mounted at /content/drive
Random seed: 3917452350


# Section 2: Dataset Paths

In [2]:
DATASETS = {
    "WELFake": "/content/drive/MyDrive/datasets/WELFake_processed.csv",
    "FakeNewsNet": "/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv",
    "Fake_News_Detection": "/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv",
    "ISOT": "/content/drive/MyDrive/datasets/ISOT_processed.csv",
    "Fake_News_Classification": "/content/drive/MyDrive/datasets/Fake_News_Classification_processed.csv",
}

PERTURBED_DATASETS = {
    "WELFake": "/content/drive/MyDrive/datasets/perturbed_outputs/WELFake_perturbed.csv",
    "FakeNewsNet": "/content/drive/MyDrive/datasets/perturbed_outputs/FakeNewsNet_perturbed.csv",
    "Fake_News_Detection": "/content/drive/MyDrive/datasets/perturbed_outputs/Fake_News_Detection_perturbed.csv",
    "ISOT": "/content/drive/MyDrive/datasets/perturbed_outputs/ISOT_perturbed.csv",
    "Fake_News_Classification": "/content/drive/MyDrive/datasets/perturbed_outputs/Fake_News_Classification_perturbed.csv",
}

# Section 3: Dataset Classes

In [3]:
class FakeNewsDataset(Dataset):
    """Standard dataset for evaluation (no perturbations)."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)


class AdversarialFakeNewsDataset(Dataset):
    """Dataset that pairs original texts with their pre-computed perturbations (pre-tokenized)."""
    def __init__(self, orig_enc, pert_enc, labels):
        self.orig_enc = orig_enc
        self.pert_enc = pert_enc
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.orig_enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        item['input_ids_pert'] = torch.tensor(self.pert_enc['input_ids'][idx], dtype=torch.long)
        item['attention_mask_pert'] = torch.tensor(self.pert_enc['attention_mask'][idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

# Section 4: AdversarialBERT Model & Loss

In [4]:
class AdversarialBERT(nn.Module):
    def __init__(self, num_labels=2, dropout=0.1):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_labels)

    def gradient_checkpointing_enable(self, **kwargs):
        self.bert.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.bert.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
        if token_type_ids is not None:
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        else:
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_emb))
        return logits, cls_emb


def adversarial_loss(logits_orig, logits_pert, labels, cls_orig, cls_pert, lambda_adv=0.5):
    ce = nn.CrossEntropyLoss()
    ce_loss = 0.5 * (ce(logits_orig, labels) + ce(logits_pert, labels))
    # Mathematically equivalent to CosineEmbeddingLoss with target=1, but avoids dynamic tensor instantiation on TPU
    cosine_sim = nn.functional.cosine_similarity(cls_orig, cls_pert, dim=-1)
    adv_loss = (1.0 - cosine_sim).mean()
    return ce_loss + (lambda_adv * adv_loss)

# Section 5: Custom Trainer & Callbacks

In [5]:
class LambdaSchedulerCallback(TrainerCallback):
    def __init__(self, max_lambda=0.5, warmup_ratio=0.1):
        self.max_lambda = max_lambda
        self.warmup_ratio = warmup_ratio
        self.trainer = None

    def on_step_begin(self, args, state, control, **kwargs):
        trainer = kwargs.get('trainer') or self.trainer
        if trainer is None:
            return
        if not hasattr(trainer, 'lambda_adv_tensor'):
            return
        total_steps = state.max_steps
        current_step = state.global_step
        if total_steps <= 0:
            return
        warmup_steps = total_steps * self.warmup_ratio
        if warmup_steps <= 0:
            new_lambda = self.max_lambda
        elif current_step < warmup_steps:
            new_lambda = self.max_lambda * (current_step / warmup_steps)
        else:
            new_lambda = self.max_lambda

        # Update the tensor in-place on TPU without JIT recompilation
        trainer.lambda_adv_tensor.fill_(new_lambda)


class AdversarialTrainer(Trainer):
    def __init__(self, *args, lambda_adv=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        # Store lambda_adv as a device tensor to prevent XLA JIT re-compilations when the float value changes
        self.lambda_adv_tensor = torch.tensor(lambda_adv, device=self.args.device, dtype=torch.float)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        input_ids_pert = inputs.get('input_ids_pert')
        attn_mask_pert = inputs.get('attention_mask_pert')

        combined_input_ids = torch.cat([inputs['input_ids'], input_ids_pert], dim=0)
        combined_attention_mask = torch.cat([inputs['attention_mask'], attn_mask_pert], dim=0)

        combined_token_type_ids = None
        if 'token_type_ids' in inputs:
            combined_token_type_ids = torch.cat([inputs['token_type_ids'], inputs['token_type_ids']], dim=0)

        combined_logits, combined_cls = model(
            input_ids=combined_input_ids,
            attention_mask=combined_attention_mask,
            token_type_ids=combined_token_type_ids
        )

        batch_size = labels.size(0)
        logits_orig, logits_pert = combined_logits[:batch_size], combined_logits[batch_size:]
        cls_orig, cls_pert = combined_cls[:batch_size], combined_cls[batch_size:]

        loss = adversarial_loss(logits_orig, logits_pert, labels, cls_orig, cls_pert, self.lambda_adv_tensor)
        return (loss, (loss, logits_orig)) if return_outputs else loss

# Section 6: Helper Functions

In [6]:
def compute_metrics(pred):
    """Compute accuracy, precision, recall, F1."""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary',
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


def load_device():
    if _TORCH_XLA_AVAILABLE and xm is not None:
        try:
            device = xm.xla_device()
            print(f"✓ Using TPU: {device}")
            return device, True
        except Exception:
            pass
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("⚠ Using CPU (Training will be slow!)")
    return device, False


def save_model(model, tokenizer, output_path, use_tpu, device):
    print("\n" + "=" * 60)
    print("MODEL SAVING")
    print("=" * 60)

    os.makedirs(output_path, exist_ok=True)

    if use_tpu:
        model.to("cpu")
        print("Moved model to CPU for saving.")

    torch.save(model.state_dict(), os.path.join(output_path, "adversarial_bert.pt"))
    tokenizer.save_pretrained(output_path)
    print(f"✓ Model saved to {output_path}")

    if use_tpu:
        model.to(device)
        print("Moved model back to TPU.")

    print("\n" + "=" * 60)
    print("ADVERSARIAL BERT FINE-TUNING COMPLETE! 🎉")
    print("=" * 60)

# Section 7: Adversarial Cross-Validation

In [7]:
def cross_validate_adversarial(full_train_texts, full_train_labels,
                               full_train_pert_texts, test_dataset,
                               tokenizer, compute_metrics_fn,
                               use_tpu, device, n_splits=5):
    print("\n" + "=" * 60)
    print(f"STARTING {n_splits}-FOLD CROSS VALIDATION (Adversarial BERT)")
    print("=" * 60)

    # Pre-tokenize full training set once to save time and RAM
    print("Pre-tokenizing full training pool...")
    full_train_enc = tokenizer(full_train_texts, truncation=True, padding='max_length', max_length=128)
    full_train_pert_enc = tokenizer(full_train_pert_texts, truncation=True, padding='max_length', max_length=128)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    indices = np.arange(len(full_train_labels))

    fold_accuracies = []
    fold_f1_scores = []
    fold_test_results = []

    # Store only the state dict of the best model on CPU to avoid OOM
    best_f1 = -1.0
    best_model_state = None
    best_fold_idx = -1

    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, full_train_labels)):
        print(f"\n--- FOLD {fold + 1} ---")

        # Slice pre-tokenized inputs for training & validation folds
        train_enc = {k: [v[i] for i in train_idx] for k, v in full_train_enc.items()}
        val_enc = {k: [v[i] for i in val_idx] for k, v in full_train_enc.items()}

        train_pert_enc = {k: [v[i] for i in train_idx] for k, v in full_train_pert_enc.items()}
        val_pert_enc = {k: [v[i] for i in val_idx] for k, v in full_train_pert_enc.items()}

        train_labels = [full_train_labels[i] for i in train_idx]
        val_labels = [full_train_labels[i] for i in val_idx]

        # Create adversarial datasets using pre-tokenized features
        train_dataset = AdversarialFakeNewsDataset(train_enc, train_pert_enc, train_labels)
        val_dataset = AdversarialFakeNewsDataset(val_enc, val_pert_enc, val_labels)

        # Fresh model each fold
        model = AdversarialBERT()
        if not use_tpu:
            model.to(device)

        training_kwargs = {
            "output_dir": f'./results_adv_fold_{fold+1}',
            "num_train_epochs": 3,
            "per_device_train_batch_size": 32,
            "per_device_eval_batch_size": 32,
            "gradient_accumulation_steps": 2,
            "save_strategy": "steps",
            "save_steps": 128,
            "save_total_limit": 1,               # Conserve disk space
            "bf16": use_tpu,
            "gradient_checkpointing": not use_tpu,
            "report_to": "none",
            "optim": "adamw_torch",
            "logging_dir": './logs',
            "logging_steps": 128,
            "metric_for_best_model": "f1",
            "load_best_model_at_end": True,
            "weight_decay": 0.01,
            "remove_unused_columns": False,
            "label_names": ["labels"],
        }
        if "evaluation_strategy" in TrainingArguments.__init__.__code__.co_varnames:
            training_kwargs["evaluation_strategy"] = "steps"
            training_kwargs["eval_steps"] = 128
        else:
            training_kwargs["eval_strategy"] = "steps"
            training_kwargs["eval_steps"] = 128

        training_args = TrainingArguments(**training_kwargs)

        lambda_cb = LambdaSchedulerCallback(max_lambda=0.5, warmup_ratio=0.1)
        trainer = AdversarialTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics_fn,
            callbacks=[
                EarlyStoppingCallback(early_stopping_patience=3),
                lambda_cb
            ],
            lambda_adv=0.0
        )
        lambda_cb.trainer = trainer

        trainer.train()

        # Validation evaluation
        eval_metrics = trainer.evaluate()
        val_f1 = eval_metrics['eval_f1']
        fold_f1_scores.append(val_f1)
        fold_accuracies.append(eval_metrics['eval_accuracy'])
        print(f"Fold {fold+1} Validation - Accuracy: {eval_metrics['eval_accuracy']:.4f}, F1: {val_f1:.4f}")

        # Save best state dict to CPU
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            best_fold_idx = fold

        # Test set evaluation using PyTorch native inference to avoid compiling new Trainer graphs
        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            test_loader = DataLoader(test_dataset, batch_size=32)
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels']

                logits, _ = model(input_ids=input_ids, attention_mask=attention_mask)
                preds = logits.argmax(-1).cpu().numpy()
                all_preds.extend(preds)
                all_targets.extend(labels.numpy())

        precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='binary')
        acc = accuracy_score(all_targets, all_preds)
        test_metrics = {
            'eval_accuracy': acc,
            'eval_f1': f1,
            'eval_precision': precision,
            'eval_recall': recall
        }
        fold_test_results.append(test_metrics)
        print(f"Fold {fold+1} Test     - Accuracy: {test_metrics['eval_accuracy']:.4f}, F1: {test_metrics['eval_f1']:.4f}, Precision: {test_metrics['eval_precision']:.4f}, Recall: {test_metrics['eval_recall']:.4f}")

        # Clean up memory immediately to prevent RAM OOM
        del model, trainer, train_dataset, val_dataset
        import gc
        gc.collect()
        if not use_tpu and torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ---- Summary ----
    print("\n" + "=" * 60)
    print("CROSS-VALIDATION SUMMARY")
    print("=" * 60)
    print(f"{'Fold':<6} {'Val Acc':<10} {'Val F1':<10} {'Test Acc':<10} {'Test F1':<10} {'Test Prec':<10} {'Test Rec':<10}")
    print("-" * 66)
    for i in range(n_splits):
        t = fold_test_results[i]
        print(f"{i+1:<6} {fold_accuracies[i]:<10.4f} {fold_f1_scores[i]:<10.4f} {t['eval_accuracy']:<10.4f} {t['eval_f1']:<10.4f} {t['eval_precision']:<10.4f} {t['eval_recall']:<10.4f}")

    avg_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    avg_f1 = np.mean(fold_f1_scores)
    std_f1 = np.std(fold_f1_scores)

    print(f"\nAverage Validation Accuracy: {avg_acc:.4f} (Std Dev: {std_acc:.4f})")
    print(f"Average Validation F1-score: {avg_f1:.4f} (Std Dev: {std_f1:.4f})")

    best_test = fold_test_results[best_fold_idx]
    print(f"\n★ Best Fold: {best_fold_idx + 1} (Val F1: {fold_f1_scores[best_fold_idx]:.4f})")
    print(f"  Test Results — Accuracy: {best_test['eval_accuracy']:.4f}, F1: {best_test['eval_f1']:.4f}, Precision: {best_test['eval_precision']:.4f}, Recall: {best_test['eval_recall']:.4f}")

    # Re-instantiate the best model on the appropriate device
    best_model = AdversarialBERT()
    best_model.load_state_dict(best_model_state)
    if not use_tpu:
        best_model.to(device)

    return best_model

# Section 8: Cross-Dataset Evaluation

In [8]:
def cross_dataset_evaluation(model, tokenizer, current_dataset_name, all_datasets_paths, compute_metrics_fn):
    """Evaluate model on all datasets except the one it was trained on.
    Outputs Accuracy, F1, Precision, and Recall for each dataset."""
    device, use_tpu = load_device()

    print("\n" + "!" * 60)
    print(f"CROSS-DATASET GENERALIZATION: {current_dataset_name}")
    print("!" * 60)

    class ModelWrapper(nn.Module):
        def __init__(self, inner_model):
            super().__init__()
            self.inner_model = inner_model
        def forward(self, input_ids, attention_mask, labels=None, **kwargs):
            outputs = self.inner_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs[0] if isinstance(outputs, (tuple, list)) else outputs.logits
            loss = None
            if labels is not None:
                loss = nn.CrossEntropyLoss()(logits, labels)
            return SequenceClassifierOutput(loss=loss, logits=logits)

    eval_model = ModelWrapper(model)

    results = {}

    for name, path in all_datasets_paths.items():
        if name == current_dataset_name:
            continue

        print(f"\nTesting on unseen dataset: {name} (Full Dataset)...")
        df = pd.read_csv(path).dropna()

        test_texts = df['combined_text'].tolist()
        test_labels = df['label'].tolist()

        encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)
        dataset = FakeNewsDataset(encodings, test_labels)

        eval_trainer = Trainer(
            model=eval_model,
            compute_metrics=compute_metrics_fn,
            args=TrainingArguments(
                output_dir="./temp_eval",
                remove_unused_columns=False,
                label_names=["labels"],
                per_device_eval_batch_size=32,
                report_to="none"
            )
        )

        metrics = eval_trainer.evaluate(eval_dataset=dataset)
        results[name] = metrics

        acc = metrics.get('eval_accuracy', 0)
        f1 = metrics.get('eval_f1', 0)
        prec = metrics.get('eval_precision', 0)
        rec = metrics.get('eval_recall', 0)

        print(f"  -> {name} Accuracy: {acc:.4f}, F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

    return results

# Section 9: Main Training Loop

In [9]:
def adversarial_train_loop(dataset_name, output_path, n_splits=5):
    dataset_path = DATASETS[dataset_name]
    perturbed_path = PERTURBED_DATASETS[dataset_name]

    # Data Loading
    print(f"\nInitialising adversarial experiment on: {dataset_name}")
    df = pd.read_csv(dataset_path).dropna().reset_index(drop=True)
    print(f"✓ Original dataset loaded: {len(df)} rows")
    print(f"Label distribution:\n{df['label'].value_counts()}")

    # Load pre-computed perturbations
    df_pert = pd.read_csv(perturbed_path).dropna().reset_index(drop=True)
    assert len(df) == len(df_pert), (
        f"Row count mismatch: original={len(df)}, perturbed={len(df_pert)}"
    )
    assert 'combined_text_perturbed' in df_pert.columns, (
        "Perturbed dataset must contain 'combined_text_perturbed' column"
    )
    print(f"✓ Perturbed dataset loaded: {len(df_pert)} rows")

    all_texts = df['combined_text'].tolist()
    all_pert_texts = df_pert['combined_text_perturbed'].tolist()
    all_labels = df['label'].tolist()

    # 85-15 split
    print("\n" + "=" * 60)
    print("TRAIN / TEST SPLIT (85-15)")
    print("=" * 60)

    indices = list(range(len(df)))
    train_idx, test_idx = train_test_split(
        indices,
        test_size=0.15,
        random_state=42,
        stratify=all_labels
    )

    train_texts = [all_texts[i] for i in train_idx]
    train_pert_texts = [all_pert_texts[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]

    test_texts = [all_texts[i] for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]

    print(f"✓ Training pool samples: {len(train_texts)} (85%)")
    print(f"✓ Test set samples:      {len(test_texts)} (15%)")

    # Tokenize test set (standard, no perturbations needed)
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    print("\nTokenizing held-out test set...")
    test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)
    test_dataset = FakeNewsDataset(test_encodings, test_labels)
    print("✓ Test set tokenized")

    # Cross-Validation
    device, use_tpu = load_device()
    best_model = cross_validate_adversarial(
        train_texts, train_labels, train_pert_texts,
        test_dataset, tokenizer, compute_metrics,
        use_tpu, device, n_splits=n_splits
    )

    # Save best model
    save_model(best_model, tokenizer, output_path, use_tpu, device)

    # Cross-dataset generalisation
    cross_dataset_evaluation(best_model, tokenizer, dataset_name, DATASETS, compute_metrics)

# WELFake Dataset

In [ ]:
adversarial_train_loop("WELFake", "/content/drive/MyDrive/models/AdversarialBERT_WELFake")


Initialising adversarial experiment on: WELFake
✓ Original dataset loaded: 63670 rows
Label distribution:
label
0    34790
1    28880
Name: count, dtype: int64
✓ Perturbed dataset loaded: 63670 rows

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 54119 (85%)
✓ Test set samples:      9551 (15%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/tmp/ipykernel_414/1195657656.py:20: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> FakeNewsNet Accuracy: 0.6996, F1: 0.8191, Precision: 0.7520, Recall: 0.8995

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.2178, F1: 0.3562, Precision: 0.3246, Recall: 0.3946

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [13]:
# 1. Determine device (TPU / GPU / CPU)
device, use_tpu = load_device()

# 2. Re-instantiate the AdversarialBERT architecture
best_model = AdversarialBERT()

# 3. Load the saved weights to CPU first to handle XLA storage tagging
model_dir = "/content/drive/MyDrive/models/AdversarialBERT_WELFake"
model_path = f"{model_dir}/adversarial_bert.pt"

print("Loading state dict onto CPU...")
# map_location="cpu" strips the xla:0 tag and safely loads tensors onto CPU RAM
state_dict = torch.load(model_path, map_location="cpu")
best_model.load_state_dict(state_dict)

# 4. Move the model to the target device (TPU/GPU) and set to evaluation mode
best_model.to(device)
best_model.eval()

# 5. Load the saved tokenizer
tokenizer = BertTokenizer.from_pretrained(model_dir)
print("✓ WELFake Model and Tokenizer successfully loaded from Google Drive!")

# 6. Conduct Cross-Dataset Evaluation
cross_dataset_evaluation(
    model=best_model,
    tokenizer=tokenizer,
    current_dataset_name="WELFake",
    all_datasets_paths=DATASETS,
    compute_metrics_fn=compute_metrics
)


/tmp/ipykernel_446/1195657656.py:20: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading state dict onto CPU...
✓ WELFake Model and Tokenizer successfully loaded from Google Drive!
✓ Using TPU: xla:0

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CROSS-DATASET GENERALIZATION: WELFake
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9982, F1: 0.9981, Precision: 0.9994, Recall: 0.9968

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0196, F1: 0.0359, Precision: 0.0383, Recall: 0.0338


{'FakeNewsNet': {'eval_loss': 1.1015338897705078,
  'eval_model_preparation_time': 0.0019,
  'eval_accuracy': 0.6995971433803333,
  'eval_f1': 0.8191489361702128,
  'eval_precision': 0.7519987855480215,
  'eval_recall': 0.8994673768308922,
  'eval_runtime': 79.8315,
  'eval_samples_per_second': 68.444,
  'eval_steps_per_second': 2.142},
 'Fake_News_Detection': {'eval_loss': 4.056006908416748,
  'eval_model_preparation_time': 0.0022,
  'eval_accuracy': 0.21777490297542043,
  'eval_f1': 0.3561830533018165,
  'eval_precision': 0.32458761886279835,
  'eval_recall': 0.3945928092856469,
  'eval_runtime': 141.9448,
  'eval_samples_per_second': 68.083,
  'eval_steps_per_second': 2.128},
 'ISOT': {'eval_loss': 0.005005216225981712,
  'eval_model_preparation_time': 0.0023,
  'eval_accuracy': 0.998235203846744,
  'eval_f1': 0.9980703079117376,
  'eval_precision': 0.9993839260711286,
  'eval_recall': 0.9967601385320076,
  'eval_runtime': 135.279,
  'eval_samples_per_second': 72.265,
  'eval_steps_

# FakeNewsNet Dataset

In [ ]:
adversarial_train_loop("FakeNewsNet", "/content/drive/MyDrive/models/AdversarialBERT_FakeNewsNet")


Initialising adversarial experiment on: FakeNewsNet
✓ Original dataset loaded: 21844 rows
Label distribution:
label
1    16522
0     5322
Name: count, dtype: int64
✓ Perturbed dataset loaded: 21844 rows

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 18567 (85%)
✓ Test set samples:      3277 (15%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  -> Fake_News_Detection Accuracy: 0.5174, F1: 0.6229, Precision: 0.5450, Recall: 0.7268

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.5246, F1: 0.5855, Precision: 0.4873, Recall: 0.7332

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.4738, F1: 0.5728, Precision: 0.5101, Recall: 0.6530


# Fake News Detection Dataset

In [ ]:
adversarial_train_loop("Fake_News_Detection", "/content/drive/MyDrive/models/AdversarialBERT_Fake_News_Detection")


Initialising adversarial experiment on: Fake_News_Detection
✓ Original dataset loaded: 38650 rows
Label distribution:
label
1    21194
0    17456
Name: count, dtype: int64
✓ Perturbed dataset loaded: 38650 rows

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 32852 (85%)
✓ Test set samples:      5798 (15%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  -> FakeNewsNet Accuracy: 0.3588, F1: 0.3132, Precision: 0.8249, Recall: 0.1933

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.0020, F1: 0.0027, Precision: 0.0025, Recall: 0.0029

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.9805, F1: 0.9817, Precision: 0.9939, Recall: 0.9698


# ISOT Dataset

In [10]:
adversarial_train_loop("ISOT", "/content/drive/MyDrive/models/AdversarialBERT_ISOT")


Initialising adversarial experiment on: ISOT
✓ Original dataset loaded: 39098 rows
Label distribution:
label
0    21196
1    17902
Name: count, dtype: int64
✓ Perturbed dataset loaded: 39098 rows

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 33233 (85%)
✓ Test set samples:      5865 (15%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  -> FakeNewsNet Accuracy: 0.7526, F1: 0.8588, Precision: 0.7557, Recall: 0.9943

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.4736, F1: 0.6428, Precision: 0.5119, Recall: 0.8636

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0185, F1: 0.0363, Precision: 0.0386, Recall: 0.0342


# Fake News Classification Dataset

In [ ]:
adversarial_train_loop("Fake_News_Classification", "/content/drive/MyDrive/models/AdversarialBERT_Fake_News_Classification")


Initialising adversarial experiment on: Fake_News_Classification
✓ Original dataset loaded: 40580 rows
Label distribution:
label
1    21923
0    18657
Name: count, dtype: int64
✓ Perturbed dataset loaded: 40580 rows

TRAIN / TEST SPLIT (85-15)
✓ Training pool samples: 34493 (85%)
✓ Test set samples:      6087 (15%)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  -> FakeNewsNet Accuracy: 0.2773, F1: 0.1015, Precision: 0.8503, Recall: 0.0540

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.5869, F1: 0.3958, Precision: 0.9996, Recall: 0.2468

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.0005, F1: 0.0002, Precision: 0.0001, Recall: 0.0002
